# BiomedCLIP + MedGemma — 13-Label CXR Multi-Label Classifier

## Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│  TRAINING                                                       │
│  image → MedGemma (frozen) → pseudo_report (text)              │
│  image → BiomedCLIP image encoder → img_emb (512-d)            │
│  pseudo_report → BiomedCLIP text encoder → txt_emb (512-d)     │
│  concat(img_emb, txt_emb) → fused_emb (1024-d)                 │
│  fused_emb → MLP → 13-label logits                             │
│  Phase 1: BiomedCLIP frozen, MLP trains                        │
│  Phase 2: End-to-end fine-tune BiomedCLIP + MLP                │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│  INFERENCE                                                      │
│  image → MedGemma → pseudo_report                              │
│  (image, pseudo_report) → BiomedCLIP → fused_emb               │
│  fused_emb → MLP → sigmoid → 13 binary predictions             │
└─────────────────────────────────────────────────────────────────┘
```

**13 CheXpert Labels:** atelectasis · cardiomegaly · consolidation · edema · enlarged cardiomediastinum · fracture · lung lesion · lung opacity · pleural effusion · pleural other · pneumonia · pneumothorax · support devices

**Metrics:** Macro F1 · Micro F1 · Hamming Accuracy · Sensitivity · Specificity · Youden-J · ROC-AUC

# Cell 0 — Configuration

In [1]:
# ============================================================
# 0) Configuration
# ============================================================
from __future__ import annotations
from pathlib import Path

# -------- Model IDs --------
BIOCLIP_MODEL_ID  = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
MEDGEMMA_MODEL_ID = "unsloth/medgemma-4b-it-bnb-4bit"
HF_TOKEN_PATH     = Path("/data/liangz2/openi/hf_token.txt")

# -------- Data paths (adjust train / val splits as needed) --------
# Expects JSONL with fields: image_path, labels (list), FINDINGS, IMPRESSION, normal
TRAIN_JSONL = Path("/data/liangz2/openi/faiss_train_mimic_biomedclip/metadata.jsonl")
VAL_JSONL   = Path("/data/liangz2/openi/faiss_val_mimic_biomedclip/metadata.jsonl")

# -------- Cache & output directories --------
CACHE_DIR              = Path("/data/liangz2/openi/biomedclip_mimic_13label_cache")
PSEUDO_REPORT_CACHE    = CACHE_DIR / "pseudo_reports.jsonl"
FUSED_EMB_TRAIN_CACHE  = CACHE_DIR / "fused_emb_train.npz"
FUSED_EMB_VAL_CACHE    = CACHE_DIR / "fused_emb_val.npz"
MODEL_SAVE_PATH_P1     = CACHE_DIR / "mlp_classifier_phase1_best.pt"
MODEL_SAVE_PATH_P2     = CACHE_DIR / "mlp_classifier_phase2_best.pt"
METRICS_CSV            = CACHE_DIR / "val_metrics_per_label.csv"

# -------- Training hyperparameters --------
BATCH_SIZE     = 64
LR_PHASE1      = 1e-3
LR_PHASE2      = 5e-5
EPOCHS_PHASE1  = 20
EPOCHS_PHASE2  = 10
DROPOUT        = 0.3
WEIGHT_DECAY   = 1e-4
THRESHOLD      = 0.5     # sigmoid threshold for binary prediction
GRAD_CLIP      = 1.0     # gradient clipping (Phase 2 only)

# -------- MedGemma generation --------
MAX_INPUT_LEN   = 4096
MAX_NEW_TOKENS  = 300
DO_SAMPLE       = False

# -------- 13 CheXpert labels (order matters — must match ground-truth encoding) --------
LABELS_13 = [
    "atelectasis",
    "cardiomegaly",
    "consolidation",
    "edema",
    "enlarged cardiomediastinum",
    "fracture",
    "lung lesion",
    "lung opacity",
    "pleural effusion",
    "pleural other",
    "pneumonia",
    "pneumothorax",
    "support devices",
]
NUM_LABELS = len(LABELS_13)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Config ready.  NUM_LABELS:", NUM_LABELS)
print("CACHE_DIR     :", CACHE_DIR)
print("TRAIN_JSONL   :", TRAIN_JSONL)
print("VAL_JSONL     :", VAL_JSONL)


Config ready.  NUM_LABELS: 13
CACHE_DIR     : /data/liangz2/openi/biomedclip_mimic_13label_cache
TRAIN_JSONL   : /data/liangz2/openi/faiss_train_mimic_biomedclip/metadata.jsonl
VAL_JSONL     : /data/liangz2/openi/faiss_val_mimic_biomedclip/metadata.jsonl


In [2]:
# ============================================================
# 1) Imports
# ============================================================
import csv
import json
import re
import time
from collections import defaultdict
from contextlib import nullcontext
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import open_clip
from huggingface_hub import login
from transformers import AutoProcessor, AutoConfig

try:
    from transformers import AutoModelForImageTextToText as _MedGemmaModel
except ImportError:
    from transformers import AutoModelForCausalLM as _MedGemmaModel

from peft import PeftModel  # kept for optional LoRA compatibility

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    hamming_loss,
    multilabel_confusion_matrix,
)

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x  # noqa: E731

# RadGraph AdamW compatibility patch (from template)
import transformers
from torch.optim import AdamW as _TorchAdamW
if not hasattr(transformers, "AdamW"):
    transformers.AdamW = _TorchAdamW

print("✅ Imports ready.")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


✅ Imports ready.
PyTorch: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [3]:
# ============================================================
# 2) HF Login + Load MedGemma backbone (for pseudo-report generation)
# ============================================================
if HF_TOKEN_PATH.exists():
    with open(HF_TOKEN_PATH, "r") as f:
        _hf_token = f.readline().strip()
    if _hf_token:
        login(token=_hf_token)
        print("✅ Hugging Face login successful.")
    else:
        print("⚠️  HF token file is empty — skipping login.")
else:
    print(f"⚠️  HF token file not found: {HF_TOKEN_PATH}")

_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID, trust_remote_code=True)
medgemma_tokenizer = (
    medgemma_processor.tokenizer
    if hasattr(medgemma_processor, "tokenizer")
    else medgemma_processor
)
if getattr(medgemma_tokenizer, "pad_token", None) is None:
    medgemma_tokenizer.pad_token = medgemma_tokenizer.eos_token

medgemma_backbone = _MedGemmaModel.from_pretrained(
    MEDGEMMA_MODEL_ID,
    trust_remote_code=True,
    dtype=_dtype,
    device_map="auto",
)
medgemma_backbone.eval()

def _medgemma_device() -> torch.device:
    try:
        return next(medgemma_backbone.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ MedGemma backbone loaded:", type(medgemma_backbone).__name__)
print("   Device:", _medgemma_device())


✅ Hugging Face login successful.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✅ MedGemma backbone loaded: Gemma3ForConditionalGeneration
   Device: cuda:0


In [4]:
# ============================================================
# 3) Load BiomedCLIP (image encoder + text encoder + tokenizer)
# ============================================================
_bioclip_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

biomedclip_model, biomedclip_preprocess = open_clip.create_model_from_pretrained(BIOCLIP_MODEL_ID)
biomedclip_tokenizer = open_clip.get_tokenizer(BIOCLIP_MODEL_ID)
biomedclip_model = biomedclip_model.to(_bioclip_device).eval()

# Resolve embedding dimension from the visual tower
EMBED_DIM = 512   # typically 512 for ViT-B/16
FUSED_DIM = EMBED_DIM * 2                        # 1024 after concat(img_emb, txt_emb)

print(f"✅ BiomedCLIP loaded on {_bioclip_device}")
print(f"   Image/Text embed dim : {EMBED_DIM}")
print(f"   Fused embed dim      : {FUSED_DIM}")

# ---- Sanity check embedding shapes ----
with torch.no_grad():
    _dummy_img  = torch.zeros(1, 3, 224, 224).to(_bioclip_device)
    _dummy_tok  = biomedclip_tokenizer(["normal chest"]).to(_bioclip_device)
    _img_feat   = biomedclip_model.encode_image(_dummy_img)
    _txt_feat   = biomedclip_model.encode_text(_dummy_tok)
print(f"   image feat shape     : {_img_feat.shape}")
print(f"   text  feat shape     : {_txt_feat.shape}")


✅ BiomedCLIP loaded on cuda
   Image/Text embed dim : 512
   Fused embed dim      : 1024
   image feat shape     : torch.Size([1, 512])
   text  feat shape     : torch.Size([1, 512])


In [5]:
# ============================================================
# 4) MedGemma pseudo-report generation
#
#    Message format: multi-modal content list
#      system  → [{"type": "text",  "text": <system_prompt>}]
#      user    → [{"type": "image", "image": <PIL.Image>},
#                 {"type": "text",  "text": <user_prompt>}]
#
#    apply_chat_template handles <start_of_image> token insertion
#    automatically when the image is in the content list — no
#    manual BOI injection needed.
#
#    Output format requested from MedGemma:
#      FINDINGS:         descriptive narrative (label names woven in naturally)
#      IMPRESSION:       overall diagnostic conclusion
#      PREDICTED LABELS: comma-separated positive labels, or
#                        "Normal chest X-ray." if all 13 are negative
# ============================================================

# ── Generation knobs ─────────────────────────────────────────────
TEMPERATURE = 0.2
TOP_P       = 0.95

# ── System prompt ────────────────────────────────────────────────
MEDGEMMA_PSEUDO_SYSTEM = (
    "You are an expert radiologist specializing in chest X-ray interpretation. "
    "Provide accurate, concise radiology reports in plain text. "
    "Do not output JSON, bullet points, markdown, or any extra commentary."
)

# ── Prompt builder ────────────────────────────────────────────────
_LABELS_BULLET = "\n".join(f"  - {lab}" for lab in LABELS_13)

def build_pseudo_report_messages(image: Image.Image) -> List[Dict[str, Any]]:
    """
    Build the multi-modal message list for MedGemma pseudo-report generation.
    The PIL image is embedded directly in the user content so
    apply_chat_template inserts the image token at the correct position.
    """
    user_text = (
        "Generate a structured radiology report for this chest X-ray.\n\n"
        "Carefully assess the image for the following 13 pathological conditions "
        "(the image may show none, one, or several):\n"
        f"{_LABELS_BULLET}\n\n"
        "Return your response in EXACTLY this format "
        "(no bullet points, no markdown, plain text only):\n\n"
        "FINDINGS: <describe all visible findings including cardiomediastinal contour, "
        "lung fields, pleural spaces, osseous structures, and any support devices; "
        "for every condition from the 13 above that is present, include its exact name "
        "naturally within the description>\n"
        "IMPRESSION: <state the overall diagnostic conclusion, naming any detected conditions>\n"
        "PREDICTED LABELS: <comma-separated list of positive conditions from the 13 above; "
        "if none of the 13 conditions are present, write exactly: Normal chest X-ray.>"
    )
    return [
        {
            "role": "system",
            "content": [{"type": "text", "text": MEDGEMMA_PSEUDO_SYSTEM}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},   # PIL image — processor handles tokenisation
                {"type": "text",  "text": user_text},
            ],
        },
    ]


# ── Utility ───────────────────────────────────────────────────────
def model_device(model) -> torch.device:
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Main generation function ─────────────────────────────────────
@torch.inference_mode()
def generate_pseudo_report(image_path: str) -> str:
    """
    Generate a pseudo radiology report for one CXR image.

    Steps
    ─────
    1. Open image as PIL.
    2. Build multi-modal messages (image embedded in content list).
    3. apply_chat_template → prompt string with image token placed correctly.
    4. processor(text, images) → tokenised model inputs.
    5. model.generate() → decode → clean.

    Returns plain-text string:
        FINDINGS: …
        IMPRESSION: …
        PREDICTED LABELS: <label1>, <label2>  |  Normal chest X-ray.
    """
    image    = Image.open(image_path).convert("RGB")
    messages = build_pseudo_report_messages(image)

    # apply_chat_template inserts <start_of_image> automatically
    # because the image is in the message content list
    input_text = medgemma_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    dev    = model_device(medgemma_backbone)
    inputs = medgemma_processor(
        text=input_text,
        images=image,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )
    # Move tensors to model device; preserve bfloat16 for pixel values
    inputs = {k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()}
    if "pixel_values" in inputs and torch.is_tensor(inputs["pixel_values"]):
        inputs["pixel_values"] = inputs["pixel_values"].to(
            torch.bfloat16 if torch.cuda.is_available() else torch.float32
        )

    out_ids = medgemma_backbone.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        temperature=TEMPERATURE if DO_SAMPLE else None,
        top_p=TOP_P       if DO_SAMPLE else None,
        pad_token_id=medgemma_tokenizer.pad_token_id,
        eos_token_id=medgemma_tokenizer.eos_token_id,
    )

    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    text    = medgemma_tokenizer.decode(gen_ids, skip_special_tokens=True)

    # Strip echo / degenerate artifacts
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^(assistant\s*:?)+", "", text, flags=re.I).strip()
    return text


print("✅ MedGemma pseudo-report generation ready.")
print()
print("Expected output format:")
print("  FINDINGS:         <descriptive narrative with label names woven in>")
print("  IMPRESSION:       <diagnostic conclusion>")
print("  PREDICTED LABELS: <label1>, <label2>  OR  Normal chest X-ray.")


✅ MedGemma pseudo-report generation ready.

Expected output format:
  FINDINGS:         <descriptive narrative with label names woven in>
  IMPRESSION:       <diagnostic conclusion>
  PREDICTED LABELS: <label1>, <label2>  OR  Normal chest X-ray.


In [6]:
# ============================================================
# 5) Ground-truth label extraction + JSONL dataset loading
# ============================================================

def extract_gt_labels_binary(row: Dict[str, Any]) -> np.ndarray:
    """
    Convert a metadata row's 'labels' field into a 13-d binary numpy vector.
    Accepts list-of-strings or comma-joined string.
    """
    lab_raw = row.get("labels", [])
    if isinstance(lab_raw, str):
        try:
            lab_raw = json.loads(lab_raw)
        except Exception:
            lab_raw = [x.strip() for x in lab_raw.split(",") if x.strip()]
    lab_set = {str(l).strip().lower() for l in lab_raw}
    return np.array([1 if lab in lab_set else 0 for lab in LABELS_13], dtype=np.float32)


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


# ---- Load datasets ----
train_records = load_jsonl(TRAIN_JSONL)
val_records   = load_jsonl(VAL_JSONL)
print(f"Train records : {len(train_records)}")
print(f"Val   records : {len(val_records)}")

# ---- Label prevalence in training set ----
_all_gt = np.stack([extract_gt_labels_binary(r) for r in train_records])
print("\nLabel prevalence (train set):")
print(f"  {'Label':<32} {'Pos':>5} {'Neg':>5} {'Prev':>6}")
print("  " + "-" * 52)
for i, lab in enumerate(LABELS_13):
    pos = int(_all_gt[:, i].sum())
    neg = len(train_records) - pos
    print(f"  {lab:<32} {pos:>5} {neg:>5} {pos/len(train_records):>6.3f}")


Train records : 10905
Val   records : 634

Label prevalence (train set):
  Label                              Pos   Neg   Prev
  ----------------------------------------------------
  atelectasis                       2760  8145  0.253
  cardiomegaly                      2277  8628  0.209
  consolidation                      491 10414  0.045
  edema                              833 10072  0.076
  enlarged cardiomediastinum         470 10435  0.043
  fracture                           308 10597  0.028
  lung lesion                        715 10190  0.066
  lung opacity                      2519  8386  0.231
  pleural effusion                  3339  7566  0.306
  pleural other                      292 10613  0.027
  pneumonia                         1211  9694  0.111
  pneumothorax                       708 10197  0.065
  support devices                   1843  9062  0.169


In [7]:
# ============================================================
# 6) Pre-compute & cache pseudo reports (MedGemma inference)
#    Skips already-cached entries — safe to re-run after interruption.
# ============================================================

def precompute_pseudo_reports(
    records: List[Dict[str, Any]],
    cache_path: Path,
    overwrite: bool = False,
) -> Dict[str, str]:
    """
    Generate pseudo reports for all unique image paths.
    Results are appended to `cache_path` (JSONL), one entry per image.
    Returns dict: image_path -> pseudo_report_text
    """
    cache: Dict[str, str] = {}

    # Load existing cache
    if cache_path.exists() and not overwrite:
        with open(cache_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    entry = json.loads(line)
                    cache[entry["image_path"]] = entry["pseudo_report"]
        print(f"  Loaded {len(cache)} cached pseudo reports from {cache_path.name}")

    # Collect unique image paths needing generation
    seen = set()
    missing = []
    for r in records:
        ip = r.get("image_path", "")
        if ip and ip not in cache and ip not in seen:
            missing.append(r)
            seen.add(ip)

    if not missing:
        print(f"  ✅ All {len(cache)} pseudo reports already cached — nothing to generate.")
        return cache

    print(f"  Generating {len(missing)} new pseudo reports …")
    with open(cache_path, "a", encoding="utf-8") as f:
        for row in tqdm(missing, desc="MedGemma pseudo-report"):
            ip = row.get("image_path", "")
            try:
                pr = generate_pseudo_report(ip)
            except Exception as exc:
                pr = f"[ERROR: {exc}]"
                print(f"    ⚠️  {ip}: {exc}")
            cache[ip] = pr
            f.write(json.dumps({"image_path": ip, "pseudo_report": pr}, ensure_ascii=False) + "\n")
            f.flush()

    print(f"  ✅ Pseudo report cache now has {len(cache)} entries → {cache_path}")
    return cache


# Combine train + val (de-duplicated) for generation pass
_all_unique = list({r["image_path"]: r for r in train_records + val_records}.values())
print(f"Unique images to process: {len(_all_unique)}")
pseudo_report_cache = precompute_pseudo_reports(_all_unique, PSEUDO_REPORT_CACHE)

# --- Quick inspection of one generated pseudo report ---
_sample_ip = list(pseudo_report_cache.keys())[0]
print(f"\n📋 Sample pseudo report for:\n   {_sample_ip}")
print("-" * 60)
print(pseudo_report_cache[_sample_ip])
print("-" * 60)


Unique images to process: 11539
  Generating 11539 new pseudo reports …


MedGemma pseudo-report:   0%|          | 0/11539 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ✅ Pseudo report cache now has 11539 entries → /data/liangz2/openi/biomedclip_mimic_13label_cache/pseudo_reports.jsonl

📋 Sample pseudo report for:
   /vf/users/liangz2/openi/mimic_train/50000766.jpg
------------------------------------------------------------
FINDINGS: The heart size is normal. The mediastinum is normal. The lungs are clear without evidence of consolidation, edema, or pneumothorax. There is no pleural effusion. The osseous structures are unremarkable. No support devices are seen. IMPRESSION: Normal chest X-ray. PREDICTED LABELS: Normal chest X-ray
------------------------------------------------------------


In [8]:
# ============================================================
# 7) BiomedCLIP embedding helpers + pre-compute fused embeddings
#    Fused = concat( L2-norm(img_emb), L2-norm(txt_emb) )  → 1024-d
# ============================================================

@torch.inference_mode()
def encode_image_biomedclip(image_path: str, normalize: bool = True) -> np.ndarray:
    img = Image.open(image_path).convert("RGB")
    x   = biomedclip_preprocess(img).unsqueeze(0).to(_bioclip_device)
    feat = biomedclip_model.encode_image(x).float()
    if normalize:
        feat = feat / feat.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return feat[0].cpu().numpy().astype(np.float32)


@torch.inference_mode()
def encode_text_biomedclip(text: str, normalize: bool = True) -> np.ndarray:
    # BiomedCLIP context length is 77 tokens; tokenizer handles truncation
    tokens = biomedclip_tokenizer([text]).to(_bioclip_device)
    feat   = biomedclip_model.encode_text(tokens).float()
    if normalize:
        feat = feat / feat.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return feat[0].cpu().numpy().astype(np.float32)


def compute_fused_embedding(image_path: str, pseudo_report: str) -> np.ndarray:
    """Concatenate L2-normalised image & text embeddings → 1024-d fused vector."""
    img_emb = encode_image_biomedclip(image_path)   # (512,)
    txt_emb = encode_text_biomedclip(pseudo_report)  # (512,)
    return np.concatenate([img_emb, txt_emb], axis=0).astype(np.float32)  # (1024,)


def precompute_fused_embeddings(
    records: List[Dict[str, Any]],
    pseudo_cache: Dict[str, str],
    save_path: Path,
    overwrite: bool = False,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Pre-compute and cache all fused embeddings and ground-truth label vectors.
    Returns (embeddings: [N, 1024], labels: [N, 13])
    """
    if save_path.exists() and not overwrite:
        data = np.load(save_path)
        embs, labs = data["embeddings"], data["labels"]
        print(f"  ✅ Loaded cached fused embeddings {embs.shape} ← {save_path.name}")
        return embs, labs

    embeddings, label_vecs = [], []
    errors = 0
    for row in tqdm(records, desc=f"BiomedCLIP fused emb ({save_path.stem})"):
        ip     = row.get("image_path", "")
        pseudo = pseudo_cache.get(ip, "")
        try:
            fused = compute_fused_embedding(ip, pseudo)
        except Exception as exc:
            print(f"    ⚠️  {ip}: {exc}")
            fused  = np.zeros(FUSED_DIM, dtype=np.float32)
            errors += 1
        embeddings.append(fused)
        label_vecs.append(extract_gt_labels_binary(row))

    embs = np.stack(embeddings).astype(np.float32)
    labs = np.stack(label_vecs).astype(np.float32)
    np.savez(save_path, embeddings=embs, labels=labs)
    print(f"  ✅ Saved {embs.shape} → {save_path}  (errors: {errors})")
    return embs, labs


# ---- Run pre-computation ----
print("Computing train fused embeddings …")
train_embeddings, train_labels = precompute_fused_embeddings(
    train_records, pseudo_report_cache, FUSED_EMB_TRAIN_CACHE
)
print("Computing val fused embeddings …")
val_embeddings, val_labels = precompute_fused_embeddings(
    val_records, pseudo_report_cache, FUSED_EMB_VAL_CACHE
)

print(f"\nTrain embeddings : {train_embeddings.shape}  labels: {train_labels.shape}")
print(f"Val   embeddings : {val_embeddings.shape}    labels: {val_labels.shape}")


Computing train fused embeddings …


BiomedCLIP fused emb (fused_emb_train):   0%|          | 0/10905 [00:00<?, ?it/s]

  ✅ Saved (10905, 1024) → /data/liangz2/openi/biomedclip_mimic_13label_cache/fused_emb_train.npz  (errors: 0)
Computing val fused embeddings …


BiomedCLIP fused emb (fused_emb_val):   0%|          | 0/634 [00:00<?, ?it/s]

  ✅ Saved (634, 1024) → /data/liangz2/openi/biomedclip_mimic_13label_cache/fused_emb_val.npz  (errors: 0)

Train embeddings : (10905, 1024)  labels: (10905, 13)
Val   embeddings : (634, 1024)    labels: (634, 13)


In [9]:
# ============================================================
# 8) Datasets & DataLoaders
# ============================================================

class FusedEmbeddingDataset(Dataset):
    """
    Phase 1 dataset: serves pre-computed (fused_embedding, label_vector) pairs.
    BiomedCLIP does NOT run at training time — embeddings come from the cache.
    """
    def __init__(self, embeddings: np.ndarray, labels: np.ndarray):
        self.embeddings = torch.from_numpy(embeddings).float()
        self.labels     = torch.from_numpy(labels).float()

    def __len__(self) -> int:
        return len(self.embeddings)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.embeddings[idx], self.labels[idx]


class ImageTextDataset(Dataset):
    """
    Phase 2 dataset: returns raw preprocessed image tensors + tokenised pseudo
    reports for end-to-end fine-tuning through BiomedCLIP.
    """
    def __init__(
        self,
        records: List[Dict[str, Any]],
        pseudo_cache: Dict[str, str],
        preprocess,
        tokenizer,
    ):
        self.records      = records
        self.pseudo_cache = pseudo_cache
        self.preprocess   = preprocess
        self.tokenizer    = tokenizer

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        row    = self.records[idx]
        ip     = row.get("image_path", "")
        pseudo = self.pseudo_cache.get(ip, "")
        gt     = extract_gt_labels_binary(row)

        img        = Image.open(ip).convert("RGB")
        img_tensor = self.preprocess(img)                    # (3, 224, 224)
        txt_tokens = self.tokenizer([pseudo])[0]             # (context_len,)

        return img_tensor, txt_tokens, torch.from_numpy(gt)


# ---- Phase 1 dataloaders (embedding-based) ----
train_emb_ds  = FusedEmbeddingDataset(train_embeddings, train_labels)
val_emb_ds    = FusedEmbeddingDataset(val_embeddings,   val_labels)

train_emb_loader = DataLoader(
    train_emb_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True
)
val_emb_loader   = DataLoader(
    val_emb_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True
)

# ---- Phase 2 dataloaders (raw image + text tokens) ----
train_imgtext_ds = ImageTextDataset(
    train_records, pseudo_report_cache,
    biomedclip_preprocess, biomedclip_tokenizer,
)
val_imgtext_ds   = ImageTextDataset(
    val_records, pseudo_report_cache,
    biomedclip_preprocess, biomedclip_tokenizer,
)

train_imgtext_loader = DataLoader(
    train_imgtext_ds, batch_size=32, shuffle=True,
    num_workers=4, pin_memory=True
)
val_imgtext_loader   = DataLoader(
    val_imgtext_ds, batch_size=32, shuffle=False,
    num_workers=4, pin_memory=True
)

print(f"Phase 1 — train batches: {len(train_emb_loader)}  val batches: {len(val_emb_loader)}")
print(f"Phase 2 — train batches: {len(train_imgtext_loader)}  val batches: {len(val_imgtext_loader)}")


Phase 1 — train batches: 171  val batches: 10
Phase 2 — train batches: 341  val batches: 20


In [10]:
# ============================================================
# 9) Model definitions
#    MLPClassifier       — standalone head operating on fused embeddings
#    BiomedCLIPWithMLP   — end-to-end model (Phase 2 fine-tuning)
# ============================================================

class MLPClassifier(nn.Module):
    """
    Multi-label 13-class MLP head for 1024-d fused BiomedCLIP embeddings.

    Architecture:
        1024 → LayerNorm → Linear(512) → LayerNorm → GELU → Dropout
             → Linear(256) → LayerNorm → GELU → Dropout
             → Linear(13)   [raw logits; apply sigmoid externally]
    """
    def __init__(self, input_dim: int, num_classes: int = 13, dropout: float = 0.3):
        super().__init__()
        self.norm_in = nn.LayerNorm(input_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout * 0.67),

            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(self.norm_in(x))


class BiomedCLIPWithMLP(nn.Module):
    """
    End-to-end wrapper used for Phase 2 fine-tuning.

    forward(images, txt_tokens) → logits [B, 13]

    freeze_biomedclip() / unfreeze_biomedclip() control which
    parameters receive gradients.
    """
    def __init__(self, biomedclip: nn.Module, mlp: MLPClassifier):
        super().__init__()
        self.biomedclip = biomedclip
        self.mlp        = mlp

    def freeze_biomedclip(self):
        for p in self.biomedclip.parameters():
            p.requires_grad_(False)
        print("✅ BiomedCLIP frozen — only MLP trains.")

    def unfreeze_biomedclip(
        self,
        unfreeze_visual: bool = True,
        unfreeze_text:   bool = True,
    ):
        if unfreeze_visual:
            for p in self.biomedclip.visual.parameters():
                p.requires_grad_(True)
        if unfreeze_text and hasattr(self.biomedclip, "text"):
            for p in self.biomedclip.text.parameters():
                p.requires_grad_(True)
        n_trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"✅ BiomedCLIP unfrozen  (visual={unfreeze_visual}, text={unfreeze_text})")
        print(f"   Trainable params: {n_trainable:,}")

    def _encode_fused(
        self,
        images:     torch.Tensor,
        txt_tokens: torch.Tensor,
    ) -> torch.Tensor:
        img_feat = self.biomedclip.encode_image(images).float()
        txt_feat = self.biomedclip.encode_text(txt_tokens).float()
        img_feat = F.normalize(img_feat, dim=-1)
        txt_feat = F.normalize(txt_feat, dim=-1)
        return torch.cat([img_feat, txt_feat], dim=-1)  # (B, 1024)

    def forward(
        self,
        images:     torch.Tensor,
        txt_tokens: torch.Tensor,
    ) -> torch.Tensor:
        fused = self._encode_fused(images, txt_tokens)
        return self.mlp(fused)


# ---- Instantiate ----
mlp        = MLPClassifier(input_dim=FUSED_DIM, num_classes=NUM_LABELS, dropout=DROPOUT).to(_bioclip_device)
full_model = BiomedCLIPWithMLP(biomedclip_model, mlp).to(_bioclip_device)

# Freeze BiomedCLIP at the start (Phase 1)
full_model.freeze_biomedclip()

_total   = sum(p.numel() for p in full_model.parameters())
_mlp_only = sum(p.numel() for p in mlp.parameters())
print(f"\nTotal parameters     : {_total:,}")
print(f"MLP-only parameters  : {_mlp_only:,}")
print(f"MLP architecture     : {FUSED_DIM} → 512 → 256 → {NUM_LABELS}")


✅ BiomedCLIP frozen — only MLP trains.

Total parameters     : 196,565,774
MLP-only parameters  : 663,053
MLP architecture     : 1024 → 512 → 256 → 13


In [11]:
# ============================================================
# 10) Loss function with class-frequency-based positive weights
#     + shared train / eval step helpers
# ============================================================

# Compute per-label positive weights  (neg_count / pos_count)
# to counteract class imbalance in BCEWithLogitsLoss
_pos_counts = train_labels.sum(axis=0)                          # (13,)
_neg_counts = len(train_labels) - _pos_counts
_pos_weight = torch.tensor(
    _neg_counts / (_pos_counts + 1e-6), dtype=torch.float32
).to(_bioclip_device)

criterion = nn.BCEWithLogitsLoss(pos_weight=_pos_weight)

print("BCEWithLogitsLoss positive weights per label:")
print(f"  {'Label':<32} {'pos_w':>6}")
print("  " + "-" * 40)
for lab, w in zip(LABELS_13, _pos_weight.cpu().tolist()):
    print(f"  {lab:<32} {w:>6.2f}")


# ---- Shared step helpers ----

def _train_step_mlp(
    mlp_model: MLPClassifier,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """One epoch over pre-computed fused embeddings (Phase 1)."""
    mlp_model.train()
    total_loss = 0.0
    for emb, labels in tqdm(loader, desc="train", leave=False):
        emb, labels = emb.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = mlp_model(emb)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(emb)
    return total_loss / len(loader.dataset)


@torch.inference_mode()
def _eval_step_mlp(
    mlp_model: MLPClassifier,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    threshold: float = THRESHOLD,
) -> Dict[str, float]:
    """Evaluate on pre-computed embeddings; returns loss + quick metrics."""
    mlp_model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    for emb, labels in loader:
        emb, labels = emb.to(device), labels.to(device)
        logits = mlp_model(emb)
        total_loss += criterion(logits, labels).item() * len(emb)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
    all_logits = torch.cat(all_logits, 0).numpy()
    all_labels = torch.cat(all_labels, 0).numpy().astype(int)
    all_probs  = torch.sigmoid(torch.from_numpy(all_logits)).numpy()
    all_preds  = (all_probs >= threshold).astype(int)
    return {
        "loss":       total_loss / len(loader.dataset),
        "macro_f1":   float(f1_score(all_labels, all_preds, average="macro",  zero_division=0)),
        "micro_f1":   float(f1_score(all_labels, all_preds, average="micro",  zero_division=0)),
        "hamming_acc": float(1.0 - hamming_loss(all_labels, all_preds)),
    }


BCEWithLogitsLoss positive weights per label:
  Label                             pos_w
  ----------------------------------------
  atelectasis                        2.95
  cardiomegaly                       3.79
  consolidation                     21.21
  edema                             12.09
  enlarged cardiomediastinum        22.20
  fracture                          34.41
  lung lesion                       14.25
  lung opacity                       3.33
  pleural effusion                   2.27
  pleural other                     36.35
  pneumonia                          8.00
  pneumothorax                      14.40
  support devices                    4.92


In [12]:
# ============================================================
# 11) Phase 1 — Train MLP with BiomedCLIP frozen
#     Input: cached 1024-d fused embeddings
#     Goal : learn the 13-label mapping efficiently
# ============================================================
optimizer_p1 = torch.optim.AdamW(
    mlp.parameters(), lr=LR_PHASE1, weight_decay=WEIGHT_DECAY
)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=EPOCHS_PHASE1, eta_min=LR_PHASE1 * 0.01
)

best_p1_macro_f1 = -1.0
train_history: List[Dict[str, Any]] = []

print("=" * 70)
print("PHASE 1  —  BiomedCLIP frozen, training MLP only")
print(f"  Epochs: {EPOCHS_PHASE1}  |  LR: {LR_PHASE1}  |  Batch: {BATCH_SIZE}")
print("=" * 70)

_t0 = time.time()
for epoch in range(1, EPOCHS_PHASE1 + 1):
    tr_loss  = _train_step_mlp(mlp, train_emb_loader, optimizer_p1, criterion, _bioclip_device)
    val_m    = _eval_step_mlp(mlp, val_emb_loader, criterion, _bioclip_device)
    scheduler_p1.step()
    lr_now   = scheduler_p1.get_last_lr()[0]

    row = {"epoch": epoch, "phase": 1, "train_loss": tr_loss, "lr": lr_now, **{f"val_{k}": v for k, v in val_m.items()}}
    train_history.append(row)

    print(
        f"[P1 {epoch:02d}/{EPOCHS_PHASE1}]  "
        f"tr_loss={tr_loss:.4f}  val_loss={val_m['loss']:.4f}  "
        f"macro_f1={val_m['macro_f1']:.4f}  micro_f1={val_m['micro_f1']:.4f}  "
        f"hamming={val_m['hamming_acc']:.4f}  lr={lr_now:.2e}"
    )

    if val_m["macro_f1"] > best_p1_macro_f1:
        best_p1_macro_f1 = val_m["macro_f1"]
        torch.save(
            {"mlp_state": mlp.state_dict(), "epoch": epoch, "phase": 1, "val_macro_f1": best_p1_macro_f1},
            MODEL_SAVE_PATH_P1,
        )
        print(f"  ✅  New best saved  (macro_f1={best_p1_macro_f1:.4f})  → {MODEL_SAVE_PATH_P1.name}")

_elapsed = time.time() - _t0
print(f"\nPhase 1 complete in {_elapsed/60:.1f} min.  Best val macro F1: {best_p1_macro_f1:.4f}")


PHASE 1  —  BiomedCLIP frozen, training MLP only
  Epochs: 20  |  LR: 0.001  |  Batch: 64


train:   0%|          | 0/171 [00:00<?, ?it/s]

[P1 01/20]  tr_loss=0.9981  val_loss=1.0668  macro_f1=0.2411  micro_f1=0.2866  hamming=0.6157  lr=9.94e-04
  ✅  New best saved  (macro_f1=0.2411)  → mlp_classifier_phase1_best.pt


train:   0%|          | 0/171 [00:00<?, ?it/s]

[P1 02/20]  tr_loss=0.9199  val_loss=1.0736  macro_f1=0.2404  micro_f1=0.2639  hamming=0.5275  lr=9.76e-04


...

[P1 18/20]  tr_loss=0.7446  val_loss=1.0823  macro_f1=0.2661  micro_f1=0.3016  hamming=0.6235  lr=3.42e-05


train:   0%|          | 0/171 [00:00<?, ?it/s]

[P1 19/20]  tr_loss=0.7416  val_loss=1.0987  macro_f1=0.2689  micro_f1=0.3100  hamming=0.6393  lr=1.61e-05


train:   0%|          | 0/171 [00:00<?, ?it/s]

[P1 20/20]  tr_loss=0.7392  val_loss=1.0984  macro_f1=0.2707  micro_f1=0.3117  hamming=0.6437  lr=1.00e-05
  ✅  New best saved  (macro_f1=0.2707)  → mlp_classifier_phase1_best.pt

Phase 1 complete in 0.3 min.  Best val macro F1: 0.2707


In [13]:
# ============================================================
# 12) Phase 2 — End-to-end fine-tuning: BiomedCLIP + MLP
#     Operates on raw images + tokenised pseudo reports.
#     Run this cell AFTER Phase 1 converges.
# ============================================================

# ---- Reload best Phase 1 weights ----
_ckpt = torch.load(MODEL_SAVE_PATH_P1, map_location=_bioclip_device)
mlp.load_state_dict(_ckpt["mlp_state"])
print(f"Loaded Phase 1 best weights  (epoch {_ckpt['epoch']}, macro_f1={_ckpt['val_macro_f1']:.4f})")

# ---- Unfreeze BiomedCLIP ----
full_model.unfreeze_biomedclip(unfreeze_visual=True, unfreeze_text=True)

# Use layer-wise LR: encoders get 10× lower LR than the MLP head
optimizer_p2 = torch.optim.AdamW(
    [
        {"params": biomedclip_model.visual.parameters(), "lr": LR_PHASE2 * 0.1},
        {"params": biomedclip_model.text.parameters(),   "lr": LR_PHASE2 * 0.1},
        {"params": mlp.parameters(),                     "lr": LR_PHASE2},
    ],
    weight_decay=WEIGHT_DECAY,
)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=EPOCHS_PHASE2, eta_min=LR_PHASE2 * 0.01
)

# ---- Phase 2 step helpers ----

def _train_step_e2e(
    model: BiomedCLIPWithMLP,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.train()
    total_loss = 0.0
    for imgs, txt_tokens, labels in tqdm(loader, desc="train e2e", leave=False):
        imgs       = imgs.to(device, dtype=torch.float32)
        txt_tokens = txt_tokens.to(device)
        labels     = labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs, txt_tokens)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item() * len(imgs)
    return total_loss / len(loader.dataset)


@torch.inference_mode()
def _eval_step_e2e(
    model: BiomedCLIPWithMLP,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    threshold: float = THRESHOLD,
) -> Dict[str, float]:
    model.eval()
    total_loss = 0.0
    all_logits, all_labels = [], []
    for imgs, txt_tokens, labels in loader:
        imgs       = imgs.to(device, dtype=torch.float32)
        txt_tokens = txt_tokens.to(device)
        labels     = labels.to(device)
        logits     = model(imgs, txt_tokens)
        total_loss += criterion(logits, labels).item() * len(imgs)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
    all_logits = torch.cat(all_logits, 0).numpy()
    all_labels = torch.cat(all_labels, 0).numpy().astype(int)
    all_probs  = torch.sigmoid(torch.from_numpy(all_logits)).numpy()
    all_preds  = (all_probs >= threshold).astype(int)
    return {
        "loss":        total_loss / len(loader.dataset),
        "macro_f1":    float(f1_score(all_labels, all_preds, average="macro",  zero_division=0)),
        "micro_f1":    float(f1_score(all_labels, all_preds, average="micro",  zero_division=0)),
        "hamming_acc": float(1.0 - hamming_loss(all_labels, all_preds)),
    }


# ---- Training loop ----
best_p2_macro_f1 = best_p1_macro_f1

print("=" * 70)
print("PHASE 2  —  End-to-end fine-tuning BiomedCLIP + MLP")
print(f"  Epochs: {EPOCHS_PHASE2}  |  MLP LR: {LR_PHASE2}  |  Encoder LR: {LR_PHASE2*0.1}")
print("=" * 70)

_t0 = time.time()
for epoch in range(1, EPOCHS_PHASE2 + 1):
    tr_loss = _train_step_e2e(full_model, train_imgtext_loader, optimizer_p2, criterion, _bioclip_device)
    val_m   = _eval_step_e2e(full_model, val_imgtext_loader, criterion, _bioclip_device)
    scheduler_p2.step()
    lr_now  = optimizer_p2.param_groups[-1]["lr"]

    row = {"epoch": epoch, "phase": 2, "train_loss": tr_loss, "lr": lr_now, **{f"val_{k}": v for k, v in val_m.items()}}
    train_history.append(row)

    print(
        f"[P2 {epoch:02d}/{EPOCHS_PHASE2}]  "
        f"tr_loss={tr_loss:.4f}  val_loss={val_m['loss']:.4f}  "
        f"macro_f1={val_m['macro_f1']:.4f}  micro_f1={val_m['micro_f1']:.4f}  "
        f"hamming={val_m['hamming_acc']:.4f}"
    )

    if val_m["macro_f1"] > best_p2_macro_f1:
        best_p2_macro_f1 = val_m["macro_f1"]
        torch.save(
            {
                "mlp_state":         mlp.state_dict(),
                "biomedclip_state":  biomedclip_model.state_dict(),
                "epoch":             epoch,
                "phase":             2,
                "val_macro_f1":      best_p2_macro_f1,
            },
            MODEL_SAVE_PATH_P2,
        )
        print(f"  ✅  New best saved  (macro_f1={best_p2_macro_f1:.4f})  → {MODEL_SAVE_PATH_P2.name}")

_elapsed = time.time() - _t0
print(f"\nPhase 2 complete in {_elapsed/60:.1f} min.  Best val macro F1: {best_p2_macro_f1:.4f}")


Loaded Phase 1 best weights  (epoch 20, macro_f1=0.2707)
✅ BiomedCLIP unfrozen  (visual=True, text=True)
   Trainable params: 196,565,773
PHASE 2  —  End-to-end fine-tuning BiomedCLIP + MLP
  Epochs: 10  |  MLP LR: 5e-05  |  Encoder LR: 5e-06


train e2e:   0%|          | 0/341 [00:00<?, ?it/s]

[P2 01/10]  tr_loss=0.7969  val_loss=1.1070  macro_f1=0.2718  micro_f1=0.3068  hamming=0.6327
  ✅  New best saved  (macro_f1=0.2718)  → mlp_classifier_phase2_best.pt


train e2e:   0%|          | 0/341 [00:00<?, ?it/s]

[P2 02/10]  tr_loss=0.7074  val_loss=1.1001  macro_f1=0.2683  micro_f1=0.3032  hamming=0.6147


...

[P2 08/10]  tr_loss=0.3403  val_loss=1.5764  macro_f1=0.2848  micro_f1=0.3622  hamming=0.7706


train e2e:   0%|          | 0/341 [00:00<?, ?it/s]

[P2 09/10]  tr_loss=0.3200  val_loss=1.6396  macro_f1=0.2797  micro_f1=0.3573  hamming=0.7691


train e2e:   0%|          | 0/341 [00:00<?, ?it/s]

[P2 10/10]  tr_loss=0.3076  val_loss=1.6481  macro_f1=0.2802  micro_f1=0.3602  hamming=0.7707

Phase 2 complete in 58.6 min.  Best val macro F1: 0.2895


In [14]:
# ============================================================
# 13) Comprehensive evaluation metrics
#     Macro F1 · Micro F1 · Hamming Accuracy · Exact Match
#     Per-label: Sensitivity · Specificity · Youden-J · ROC-AUC
# ============================================================

def compute_comprehensive_metrics(
    y_true:  np.ndarray,   # (N, 13) int
    y_pred:  np.ndarray,   # (N, 13) int  — threshold-binarised
    y_score: np.ndarray,   # (N, 13) float — sigmoid probabilities
    labels:  List[str] = LABELS_13,
    threshold: float   = THRESHOLD,
) -> Dict[str, Any]:
    """
    Returns a dict with:
      - summary scalars: macro_f1, micro_f1, hamming_accuracy,
                         exact_match_accuracy, macro_sensitivity,
                         macro_specificity, macro_youden_j,
                         roc_auc_macro, roc_auc_micro
      - 'per_label': dict[label → {TP,FP,TN,FN,f1,sensitivity,
                                    specificity,youden_j,roc_auc}]
    """
    N, C = y_true.shape

    # ── F1 ──────────────────────────────────────────────────────
    macro_f1      = float(f1_score(y_true, y_pred, average="macro",  zero_division=0))
    micro_f1      = float(f1_score(y_true, y_pred, average="micro",  zero_division=0))
    per_label_f1  = f1_score(y_true, y_pred, average=None, zero_division=0)

    # ── Hamming & Exact-match ────────────────────────────────────
    hamming_acc  = float(1.0 - hamming_loss(y_true, y_pred))
    exact_match  = float((y_true == y_pred).all(axis=1).mean())

    # ── Per-label confusion matrices → Sensitivity / Specificity ─
    # multilabel_confusion_matrix returns (C, 2, 2):
    # mcm[i] = [[TN, FP], [FN, TP]]
    mcm = multilabel_confusion_matrix(y_true, y_pred)

    per_label: Dict[str, Dict[str, Any]] = {}
    sens_list, spec_list, youden_list = [], [], []

    for i, lab in enumerate(labels):
        tn, fp, fn, tp = mcm[i].ravel()
        sens   = tp / (tp + fn + 1e-9) if (tp + fn) > 0 else float("nan")
        spec   = tn / (tn + fp + 1e-9) if (tn + fp) > 0 else float("nan")
        youden = (sens + spec - 1.0) if not (np.isnan(sens) or np.isnan(spec)) else float("nan")

        per_label[lab] = {
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
            "f1":          float(per_label_f1[i]),
            "sensitivity": float(sens),
            "specificity": float(spec),
            "youden_j":    float(youden),
        }
        if not np.isnan(sens):   sens_list.append(sens)
        if not np.isnan(spec):   spec_list.append(spec)
        if not np.isnan(youden): youden_list.append(youden)

    macro_sensitivity = float(np.mean(sens_list))   if sens_list   else float("nan")
    macro_specificity = float(np.mean(spec_list))   if spec_list   else float("nan")
    macro_youden      = float(np.mean(youden_list)) if youden_list else float("nan")

    # ── ROC-AUC ──────────────────────────────────────────────────
    try:
        roc_auc_macro = float(roc_auc_score(y_true, y_score, average="macro"))
        roc_auc_micro = float(roc_auc_score(y_true, y_score, average="micro"))
        per_label_auc = roc_auc_score(y_true, y_score, average=None)
        for i, lab in enumerate(labels):
            per_label[lab]["roc_auc"] = float(per_label_auc[i])
    except Exception as _e:
        roc_auc_macro = float("nan")
        roc_auc_micro = float("nan")
        print(f"⚠️  ROC-AUC skipped: {_e}")

    return {
        # ── Summary ──
        "N":                    N,
        "threshold":            threshold,
        "macro_f1":             macro_f1,
        "micro_f1":             micro_f1,
        "hamming_accuracy":     hamming_acc,
        "exact_match_accuracy": exact_match,
        "macro_sensitivity":    macro_sensitivity,
        "macro_specificity":    macro_specificity,
        "macro_youden_j":       macro_youden,
        "roc_auc_macro":        roc_auc_macro,
        "roc_auc_micro":        roc_auc_micro,
        # ── Per-label breakdown ──
        "per_label": per_label,
    }


def print_metrics_table(metrics: Dict[str, Any]) -> None:
    """Pretty-print the comprehensive metric table."""
    print("\n" + "=" * 80)
    print("  EVALUATION METRICS SUMMARY")
    print("=" * 80)
    print(f"  N samples             : {metrics['N']}")
    print(f"  Threshold             : {metrics['threshold']}")
    print(f"  Macro F1              : {metrics['macro_f1']:.4f}")
    print(f"  Micro F1              : {metrics['micro_f1']:.4f}")
    print(f"  Hamming Accuracy      : {metrics['hamming_accuracy']:.4f}")
    print(f"  Exact Match Accuracy  : {metrics['exact_match_accuracy']:.4f}")
    print(f"  Macro Sensitivity     : {metrics['macro_sensitivity']:.4f}")
    print(f"  Macro Specificity     : {metrics['macro_specificity']:.4f}")
    print(f"  Macro Youden-J        : {metrics['macro_youden_j']:.4f}")
    print(f"  ROC-AUC (macro)       : {metrics['roc_auc_macro']:.4f}")
    print(f"  ROC-AUC (micro)       : {metrics['roc_auc_micro']:.4f}")
    print()
    _H = f"  {'Label':<32} {'F1':>5} {'Sens':>5} {'Spec':>5} {'Youden':>6} {'AUC':>5}"
    print(_H)
    print("  " + "-" * (len(_H) - 2))
    for lab, m in metrics["per_label"].items():
        auc = m.get("roc_auc", float("nan"))
        print(
            f"  {lab:<32} {m['f1']:>5.3f} {m['sensitivity']:>5.3f} "
            f"{m['specificity']:>5.3f} {m['youden_j']:>6.3f} {auc:>5.3f}"
        )
    print("=" * 80 + "\n")


print("✅ Comprehensive metrics function ready.")


✅ Comprehensive metrics function ready.


In [15]:
# ============================================================
# 14) Full evaluation run on validation set
#     Loads the best saved checkpoint (Phase 2 if available,
#     else Phase 1), runs a complete forward pass, and reports
#     all metrics.
# ============================================================

def _collect_predictions_mlp(
    mlp_model: MLPClassifier,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Returns (y_true, y_pred, y_score) from a pre-computed embedding loader."""
    mlp_model.eval()
    all_logits, all_labels = [], []
    with torch.inference_mode():
        for emb, labels in tqdm(loader, desc="eval"):
            all_logits.append(mlp_model(emb.to(device)).cpu())
            all_labels.append(labels)
    all_logits = torch.cat(all_logits, 0).numpy()
    all_labels = torch.cat(all_labels, 0).numpy().astype(int)
    all_probs  = torch.sigmoid(torch.from_numpy(all_logits)).numpy()
    all_preds  = (all_probs >= THRESHOLD).astype(int)
    return all_labels, all_preds, all_probs


def _collect_predictions_e2e(
    model: BiomedCLIPWithMLP,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Returns (y_true, y_pred, y_score) from an ImageTextDataset loader."""
    model.eval()
    all_logits, all_labels = [], []
    with torch.inference_mode():
        for imgs, txt_tokens, labels in tqdm(loader, desc="eval e2e"):
            logits = model(imgs.to(device, dtype=torch.float32), txt_tokens.to(device))
            all_logits.append(logits.cpu())
            all_labels.append(labels)
    all_logits = torch.cat(all_logits, 0).numpy()
    all_labels = torch.cat(all_labels, 0).numpy().astype(int)
    all_probs  = torch.sigmoid(torch.from_numpy(all_logits)).numpy()
    all_preds  = (all_probs >= THRESHOLD).astype(int)
    return all_labels, all_preds, all_probs


# ---- Load best checkpoint ----
_use_p2 = MODEL_SAVE_PATH_P2.exists()
if _use_p2:
    _ckpt = torch.load(MODEL_SAVE_PATH_P2, map_location=_bioclip_device)
    mlp.load_state_dict(_ckpt["mlp_state"])
    if "biomedclip_state" in _ckpt:
        biomedclip_model.load_state_dict(_ckpt["biomedclip_state"])
    print(f"✅ Loaded Phase 2 best  (epoch {_ckpt['epoch']}, macro_f1={_ckpt['val_macro_f1']:.4f})")
    print("   Running e2e evaluation through BiomedCLIP …")
    y_true, y_pred, y_score = _collect_predictions_e2e(full_model, val_imgtext_loader, _bioclip_device)
else:
    _ckpt = torch.load(MODEL_SAVE_PATH_P1, map_location=_bioclip_device)
    mlp.load_state_dict(_ckpt["mlp_state"])
    print(f"✅ Loaded Phase 1 best  (epoch {_ckpt['epoch']}, macro_f1={_ckpt['val_macro_f1']:.4f})")
    print("   Running evaluation on pre-computed embeddings …")
    y_true, y_pred, y_score = _collect_predictions_mlp(mlp, val_emb_loader, _bioclip_device)

# ---- Compute and display all metrics ----
val_metrics = compute_comprehensive_metrics(y_true, y_pred, y_score)
print_metrics_table(val_metrics)


✅ Loaded Phase 2 best  (epoch 5, macro_f1=0.2895)
   Running e2e evaluation through BiomedCLIP …


eval e2e:   0%|          | 0/20 [00:00<?, ?it/s]


  EVALUATION METRICS SUMMARY
  N samples             : 634
  Threshold             : 0.5
  Macro F1              : 0.2895
  Micro F1              : 0.3477
  Hamming Accuracy      : 0.7264
  Exact Match Accuracy  : 0.2145
  Macro Sensitivity     : 0.6455
  Macro Specificity     : 0.7219
  Macro Youden-J        : 0.3675
  ROC-AUC (macro)       : 0.7457
  ROC-AUC (micro)       : 0.7814

  Label                               F1  Sens  Spec Youden   AUC
  ---------------------------------------------------------------
  atelectasis                      0.384 0.739 0.642  0.381 0.736
  cardiomegaly                     0.413 0.771 0.592  0.363 0.744
  consolidation                    0.199 0.690 0.749  0.438 0.774
  edema                            0.407 0.733 0.805  0.538 0.838
  enlarged cardiomediastinum       0.066 0.400 0.738  0.138 0.681
  fracture                         0.218 0.419 0.876  0.295 0.644
  lung lesion                      0.192 0.778 0.718  0.496 0.725
  lung opacity    

In [16]:
# ============================================================
# 15) End-to-end inference pipeline
#     predict_13_labels(image_path) → {present_labels, probs, pseudo_report}
#     Steps:
#       1. image → MedGemma → pseudo_report text
#       2. image → BiomedCLIP image encoder → img_emb (512-d)
#          pseudo_report → BiomedCLIP text encoder → txt_emb (512-d)
#       3. concat(img_emb, txt_emb) → MLP → sigmoid → 13 predictions
# ============================================================

@torch.inference_mode()
def predict_13_labels(
    image_path: str,
    threshold:  float = THRESHOLD,
    return_probs: bool = True,
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Full inference pipeline for a single chest X-ray.

    Parameters
    ----------
    image_path  : path to the input image
    threshold   : sigmoid cut-off for positive prediction (default 0.5)
    return_probs: include per-label probabilities in the output
    verbose     : print the generated pseudo report

    Returns
    -------
    dict with keys:
      image_path       – input path
      pseudo_report    – MedGemma-generated structured report
      present_labels   – list of positively predicted label names
      label_predictions – {label: 0/1}
      label_probabilities – {label: float}  (if return_probs)
    """
    # Step 1 ── MedGemma generates a pseudo report
    pseudo_report = generate_pseudo_report(image_path)
    if verbose:
        print("── Pseudo Report ──────────────────────────────────────")
        print(pseudo_report)
        print("───────────────────────────────────────────────────────")

    # Step 2 ── BiomedCLIP fused embedding
    img_emb = encode_image_biomedclip(image_path)    # (512,)
    txt_emb = encode_text_biomedclip(pseudo_report)   # (512,)
    fused   = np.concatenate([img_emb, txt_emb], axis=0)  # (1024,)

    # Step 3 ── MLP classification
    mlp.eval()
    emb_t  = torch.from_numpy(fused).unsqueeze(0).float().to(_bioclip_device)
    logits = mlp(emb_t)
    probs  = torch.sigmoid(logits)[0].cpu().numpy()
    preds  = (probs >= threshold).astype(int)

    present_labels = [LABELS_13[i] for i in range(NUM_LABELS) if preds[i] == 1]

    result: Dict[str, Any] = {
        "image_path":        image_path,
        "pseudo_report":     pseudo_report,
        "present_labels":    present_labels,
        "label_predictions": {lab: int(preds[i])  for i, lab in enumerate(LABELS_13)},
    }
    if return_probs:
        result["label_probabilities"] = {lab: float(probs[i]) for i, lab in enumerate(LABELS_13)}

    return result


# ---- Batch inference helper ----
def batch_predict_13_labels(
    image_paths: List[str],
    threshold:   float = THRESHOLD,
    progress:    bool  = True,
) -> List[Dict[str, Any]]:
    """Run predict_13_labels for a list of image paths."""
    results = []
    iterator = tqdm(image_paths, desc="Inference") if progress else image_paths
    for ip in iterator:
        try:
            r = predict_13_labels(ip, threshold=threshold)
        except Exception as exc:
            r = {"image_path": ip, "error": str(exc), "present_labels": [], "label_predictions": {}}
        results.append(r)
    return results


print("✅ Inference pipeline ready.")
print()
print("Usage:")
print("  result = predict_13_labels('/path/to/cxr.jpg')")
print("  print(result['present_labels'])          # e.g. ['atelectasis', 'pleural effusion']")
print("  print(result['pseudo_report'])            # MedGemma structured report")
print("  print(result['label_probabilities'])      # dict of per-label sigmoid scores")


✅ Inference pipeline ready.

Usage:
  result = predict_13_labels('/path/to/cxr.jpg')
  print(result['present_labels'])          # e.g. ['atelectasis', 'pleural effusion']
  print(result['pseudo_report'])            # MedGemma structured report
  print(result['label_probabilities'])      # dict of per-label sigmoid scores


In [28]:
result = predict_13_labels('/data/liangz2/openi/mimic_test/50022945.jpg')
print(result['present_labels'])          # e.g. ['atelectasis', 'pleural effusion']
print(result['pseudo_report'])            # MedGemma structured report
print(result['label_probabilities'])      # dict of per-label sigmoid scores

['atelectasis', 'cardiomegaly', 'enlarged cardiomediastinum', 'pleural effusion', 'pneumothorax', 'support devices']
FINDINGS: The heart size is normal. The cardiomediastinal silhouette is unremarkable. The lung fields are clear. There is no evidence of consolidation, edema, pleural effusion, pneumothorax, or lung lesion. The osseous structures are unremarkable. A central line is present. IMPRESSION: Normal chest X-ray. PREDICTED LABELS: Normal chest X-ray
{'atelectasis': 0.936522901058197, 'cardiomegaly': 0.6568131446838379, 'consolidation': 0.14161138236522675, 'edema': 0.10133448243141174, 'enlarged cardiomediastinum': 0.9192487597465515, 'fracture': 0.13112185895442963, 'lung lesion': 0.03132122382521629, 'lung opacity': 0.4941920042037964, 'pleural effusion': 0.9296914935112, 'pleural other': 0.03276399150490761, 'pneumonia': 0.19584350287914276, 'pneumothorax': 0.8030743598937988, 'support devices': 0.8695693612098694}


In [17]:
# ============================================================
# 16) Save metrics to CSV + threshold sweep + training curve
# ============================================================
import csv as _csv

def save_metrics_csv(metrics: Dict[str, Any], save_path: Path) -> None:
    """Write per-label metrics table + summary row to a CSV file."""
    fieldnames = ["label", "f1", "sensitivity", "specificity",
                  "youden_j", "roc_auc", "TP", "FP", "TN", "FN"]
    rows = []
    for lab, m in metrics["per_label"].items():
        rows.append({
            "label":       lab,
            "f1":          m["f1"],
            "sensitivity": m["sensitivity"],
            "specificity": m["specificity"],
            "youden_j":    m["youden_j"],
            "roc_auc":     m.get("roc_auc", float("nan")),
            "TP":          m["TP"],
            "FP":          m["FP"],
            "TN":          m["TN"],
            "FN":          m["FN"],
        })
    # Summary row
    rows.append({
        "label":       "** MACRO SUMMARY **",
        "f1":          metrics["macro_f1"],
        "sensitivity": metrics["macro_sensitivity"],
        "specificity": metrics["macro_specificity"],
        "youden_j":    metrics["macro_youden_j"],
        "roc_auc":     metrics["roc_auc_macro"],
        "TP": "", "FP": "", "TN": "", "FN": "",
    })
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", newline="", encoding="utf-8") as f:
        writer = _csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"✅ Metrics saved → {save_path}")


save_metrics_csv(val_metrics, METRICS_CSV)


# ---- Threshold sweep (find optimal threshold per macro-F1) ----
print("\nThreshold sweep (macro F1):")
print(f"  {'Threshold':>9}  {'Macro F1':>8}  {'Micro F1':>8}  {'Hamming':>8}  {'Youden-J':>9}")
print("  " + "-" * 55)
best_thr, best_f1 = THRESHOLD, -1.0
for thr in np.arange(0.2, 0.8, 0.05):
    _preds = (y_score >= thr).astype(int)
    _mf1   = float(f1_score(y_true, _preds, average="macro",  zero_division=0))
    _uf1   = float(f1_score(y_true, _preds, average="micro",  zero_division=0))
    _ham   = float(1.0 - hamming_loss(y_true, _preds))
    _m     = compute_comprehensive_metrics(y_true, _preds, y_score, threshold=thr)
    _yj    = _m["macro_youden_j"]
    marker = "  ◀ current" if abs(thr - THRESHOLD) < 0.01 else ""
    if _mf1 > best_f1:
        best_f1, best_thr = _mf1, thr
        marker += "  ★ best"
    print(f"  {thr:>9.2f}  {_mf1:>8.4f}  {_uf1:>8.4f}  {_ham:>8.4f}  {_yj:>9.4f}{marker}")

print(f"\n  ★ Best threshold: {best_thr:.2f}  →  macro F1 = {best_f1:.4f}")


# ---- Training history summary ----
if train_history:
    print("\nTraining history (last 5 epochs per phase):")
    _by_phase: Dict[int, List] = defaultdict(list)
    for row in train_history:
        _by_phase[row["phase"]].append(row)
    for phase, rows in sorted(_by_phase.items()):
        print(f"\n  Phase {phase}:")
        print(f"    {'Epoch':>5}  {'tr_loss':>8}  {'val_loss':>8}  {'macro_f1':>8}  {'micro_f1':>8}  {'hamming':>8}")
        for r in rows[-5:]:
            print(
                f"    {r['epoch']:>5}  {r['train_loss']:>8.4f}  {r['val_loss']:>8.4f}  "
                f"{r['val_macro_f1']:>8.4f}  {r['val_micro_f1']:>8.4f}  {r['val_hamming_acc']:>8.4f}"
            )

print("\n✅ All done.")


✅ Metrics saved → /data/liangz2/openi/biomedclip_mimic_13label_cache/val_metrics_per_label.csv

Threshold sweep (macro F1):
  Threshold  Macro F1  Micro F1   Hamming   Youden-J
  -------------------------------------------------------
       0.20    0.2431    0.2845    0.5772     0.3154  ★ best
       0.25    0.2513    0.2967    0.6090     0.3243  ★ best
       0.30    0.2590    0.3069    0.6355     0.3392  ★ best
       0.35    0.2671    0.3183    0.6622     0.3444  ★ best
       0.40    0.2739    0.3283    0.6837     0.3436  ★ best
       0.45    0.2850    0.3420    0.7067     0.3650  ★ best
       0.50    0.2895    0.3477    0.7264     0.3675  ◀ current  ★ best
       0.55    0.2949    0.3584    0.7490     0.3690  ★ best
       0.60    0.2976    0.3627    0.7677     0.3638  ★ best
       0.65    0.3029    0.3726    0.7876     0.3610  ★ best
       0.70    0.3075    0.3834    0.8084     0.3557  ★ best
       0.75    0.3189    0.4009    0.8314     0.3532  ★ best
       0.80    0.3059 

In [18]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import PretrainedConfig, PreTrainedModel


# ============================================================
# Custom Config
# ============================================================
class BiomedCLIPMLPConfig(PretrainedConfig):
    model_type = "biomedclip_mlp_classifier"

    def __init__(
        self,
        biomedclip_model_id: str = "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
        input_dim: int = 1024,
        image_embedding_dim: int = 512,
        text_embedding_dim: int = 512,
        hidden_dims: Optional[List[int]] = None,
        dropout: float = 0.3,
        num_labels: int = 13,
        label_names: Optional[List[str]] = None,
        threshold: float = 0.5,
        phase_exported: Optional[int] = None,
        checkpoint_source: Optional[str] = None,
        has_biomedclip_state: bool = False,
        biomedclip_state_filename: Optional[str] = None,
        problem_type: str = "multi_label_classification",
        **kwargs,
    ):
        super().__init__(**kwargs)

        if hidden_dims is None:
            hidden_dims = [512, 256]

        if label_names is None:
            label_names = [
                "atelectasis",
                "cardiomegaly",
                "consolidation",
                "edema",
                "enlarged cardiomediastinum",
                "fracture",
                "lung lesion",
                "lung opacity",
                "pleural effusion",
                "pleural other",
                "pneumonia",
                "pneumothorax",
                "support devices",
            ]

        self.biomedclip_model_id = biomedclip_model_id
        self.input_dim = input_dim
        self.image_embedding_dim = image_embedding_dim
        self.text_embedding_dim = text_embedding_dim
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.num_labels = num_labels
        self.label_names = label_names
        self.threshold = threshold
        self.phase_exported = phase_exported
        self.checkpoint_source = checkpoint_source
        self.has_biomedclip_state = has_biomedclip_state
        self.biomedclip_state_filename = biomedclip_state_filename
        self.problem_type = problem_type

        self.id2label = {i: label for i, label in enumerate(label_names)}
        self.label2id = {label: i for i, label in enumerate(label_names)}


# ============================================================
# MLP head
# ============================================================
class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: List[int], num_labels: int, dropout: float = 0.3):
        super().__init__()

        if len(hidden_dims) != 2:
            raise ValueError("This implementation expects hidden_dims to have length 2, e.g. [512, 256].")

        h1, h2 = hidden_dims

        self.norm_in = nn.LayerNorm(input_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.LayerNorm(h1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.LayerNorm(h2),
            nn.GELU(),
            nn.Dropout(dropout * 0.67),

            nn.Linear(h2, num_labels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(self.norm_in(x))


# ============================================================
# Custom HF model
# ============================================================
class BiomedCLIPMLPForMultiLabelClassification(PreTrainedModel):
    config_class = BiomedCLIPMLPConfig
    base_model_prefix = "biomedclip_mlp"

    def __init__(self, config: BiomedCLIPMLPConfig, biomedclip_model: Optional[nn.Module] = None):
        super().__init__(config)

        self.biomedclip = biomedclip_model
        self.classifier = MLPClassifier(
            input_dim=config.input_dim,
            hidden_dims=config.hidden_dims,
            num_labels=config.num_labels,
            dropout=config.dropout,
        )

        self.post_init()

    def set_biomedclip_model(self, biomedclip_model: nn.Module) -> None:
        self.biomedclip = biomedclip_model

    def encode_fused(self, images: torch.Tensor, text_tokens: torch.Tensor) -> torch.Tensor:
        if self.biomedclip is None:
            raise ValueError(
                "self.biomedclip is None. Attach a BiomedCLIP model first using "
                "`model.set_biomedclip_model(biomedclip_model)`."
            )

        img_feat = self.biomedclip.encode_image(images).float()
        txt_feat = self.biomedclip.encode_text(text_tokens).float()

        img_feat = F.normalize(img_feat, dim=-1)
        txt_feat = F.normalize(txt_feat, dim=-1)
        fused = torch.cat([img_feat, txt_feat], dim=-1)
        return fused

    def forward(
        self,
        fused_embeddings: Optional[torch.Tensor] = None,
        images: Optional[torch.Tensor] = None,
        text_tokens: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        return_dict: bool = True,
    ) -> Union[Dict[str, torch.Tensor], tuple]:
        if fused_embeddings is None:
            if images is None or text_tokens is None:
                raise ValueError(
                    "Provide either `fused_embeddings` directly, or both `images` and `text_tokens`."
                )
            fused_embeddings = self.encode_fused(images, text_tokens)

        logits = self.classifier(fused_embeddings)
        probs = torch.sigmoid(logits)

        loss = None
        if labels is not None:
            labels = labels.float()
            loss = F.binary_cross_entropy_with_logits(logits, labels)

        if not return_dict:
            output = (logits, probs)
            return ((loss,) + output) if loss is not None else output

        return {
            "loss": loss,
            "logits": logits,
            "probabilities": probs,
        }

In [19]:
# ============================================================
# Export function
# ============================================================
def export_biomedclip_mlp_as_hf_pretrained(
    output_dir: str | Path,
    phase1_ckpt_path: str | Path = "/data/liangz2/openi/biomedclip_mimic_13label_cache/mlp_classifier_phase1_best.pt",
    phase2_ckpt_path: str | Path = "/data/liangz2/openi/biomedclip_mimic_13label_cache/mlp_classifier_phase2_best.pt",
    biomedclip_model_id: str = "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    label_names: Optional[List[str]] = None,
    input_dim: int = 1024,
    image_embedding_dim: int = 512,
    text_embedding_dim: int = 512,
    hidden_dims: Optional[List[int]] = None,
    dropout: float = 0.3,
    threshold: float = 0.5,
    save_biomedclip_state_separately: bool = True,
    safe_serialization: bool = False,
) -> Path:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    phase1_ckpt_path = Path(phase1_ckpt_path)
    phase2_ckpt_path = Path(phase2_ckpt_path)

    if hidden_dims is None:
        hidden_dims = [512, 256]

    if label_names is None:
        label_names = [
            "atelectasis",
            "cardiomegaly",
            "consolidation",
            "edema",
            "enlarged cardiomediastinum",
            "fracture",
            "lung lesion",
            "lung opacity",
            "pleural effusion",
            "pleural other",
            "pneumonia",
            "pneumothorax",
            "support devices",
        ]

    if phase2_ckpt_path.exists():
        ckpt_path = phase2_ckpt_path
        phase = 2
    elif phase1_ckpt_path.exists():
        ckpt_path = phase1_ckpt_path
        phase = 1
    else:
        raise FileNotFoundError(
            f"Cannot find either checkpoint:\n{phase1_ckpt_path}\n{phase2_ckpt_path}"
        )

    ckpt = torch.load(ckpt_path, map_location="cpu")

    if "mlp_state" not in ckpt:
        raise KeyError(f"`mlp_state` not found in checkpoint: {ckpt_path}")

    has_biomedclip_state = bool(phase == 2 and "biomedclip_state" in ckpt)
    biomedclip_state_filename = "biomedclip_state.bin" if has_biomedclip_state else None

    config = BiomedCLIPMLPConfig(
        biomedclip_model_id=biomedclip_model_id,
        input_dim=input_dim,
        image_embedding_dim=image_embedding_dim,
        text_embedding_dim=text_embedding_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
        num_labels=len(label_names),
        label_names=label_names,
        threshold=threshold,
        phase_exported=phase,
        checkpoint_source=str(ckpt_path),
        has_biomedclip_state=has_biomedclip_state,
        biomedclip_state_filename=biomedclip_state_filename,
        problem_type="multi_label_classification",
    )

    model = BiomedCLIPMLPForMultiLabelClassification(config)
    model.classifier.load_state_dict(ckpt["mlp_state"])

    # Save model + config using HF API
    model.save_pretrained(output_dir, safe_serialization=safe_serialization)

    # Save BiomedCLIP backbone state separately if present
    if has_biomedclip_state and save_biomedclip_state_separately:
        torch.save(ckpt["biomedclip_state"], output_dir / biomedclip_state_filename)

    # Save detailed architecture JSON
    architecture = {
        "model_name": "BiomedCLIPMLPForMultiLabelClassification",
        "hf_class_name": "BiomedCLIPMLPForMultiLabelClassification",
        "config_class_name": "BiomedCLIPMLPConfig",
        "task": "multi-label chest X-ray pathology classification",
        "exported_phase": phase,
        "checkpoint_source": str(ckpt_path),
        "labels": label_names,
        "num_labels": len(label_names),
        "threshold": threshold,
        "backbone": {
            "name": "BiomedCLIP",
            "model_id": biomedclip_model_id,
            "image_embedding_dim": image_embedding_dim,
            "text_embedding_dim": text_embedding_dim,
            "fusion": "concat(L2_normalize(image_emb), L2_normalize(text_emb))",
            "has_saved_backbone_state": has_biomedclip_state,
            "backbone_state_filename": biomedclip_state_filename,
        },
        "classifier_head": {
            "type": "MLPClassifier",
            "input_dim": input_dim,
            "hidden_dims": hidden_dims,
            "dropout": dropout,
            "output_dim": len(label_names),
            "output": "raw logits",
            "activation_for_inference": "sigmoid",
        },
        "checkpoint_metrics": {
            "epoch": ckpt.get("epoch"),
            "train_loss": ckpt.get("train_loss"),
            "val_loss": ckpt.get("val_loss"),
            "val_macro_f1": ckpt.get("val_macro_f1"),
            "val_micro_f1": ckpt.get("val_micro_f1"),
            "val_hamming_acc": ckpt.get("val_hamming_acc"),
        },
    }

    with open(output_dir / "architecture.json", "w", encoding="utf-8") as f:
        json.dump(architecture, f, indent=2, ensure_ascii=False)

    # Save plain label list
    with open(output_dir / "label_names.json", "w", encoding="utf-8") as f:
        json.dump(label_names, f, indent=2, ensure_ascii=False)

    # Optional extra metadata for reconstruction
    processor_meta = {
        "biomedclip_model_id": biomedclip_model_id,
        "expected_inputs": {
            "option_1": "fused_embeddings: FloatTensor [B, 1024]",
            "option_2": {
                "images": "image tensor accepted by BiomedCLIP.encode_image",
                "text_tokens": "tokenized text tensor accepted by BiomedCLIP.encode_text"
            }
        },
        "prediction": {
            "logits_to_probabilities": "sigmoid",
            "default_threshold": threshold
        }
    }

    with open(output_dir / "processor_config.json", "w", encoding="utf-8") as f:
        json.dump(processor_meta, f, indent=2, ensure_ascii=False)

    print(f"✅ Hugging Face export completed: {output_dir}")
    print(f"   Files:")
    for p in sorted(output_dir.iterdir()):
        print(f"   - {p.name}")

    return output_dir

In [20]:
# ============================================================
# Load with from_pretrained
# ============================================================
def load_biomedclip_mlp_from_pretrained(
    model_dir: str | Path,
    biomedclip_model: Optional[nn.Module] = None,
    map_location: str = "cpu",
) -> BiomedCLIPMLPForMultiLabelClassification:
    model_dir = Path(model_dir)

    model = BiomedCLIPMLPForMultiLabelClassification.from_pretrained(
        model_dir,
        local_files_only=True,
    )

    model.to(map_location)
    model.eval()

    if biomedclip_model is not None:
        model.set_biomedclip_model(biomedclip_model)

    # Load saved backbone state if available and if a backbone model is supplied
    config = model.config
    if (
        biomedclip_model is not None
        and getattr(config, "has_biomedclip_state", False)
        and getattr(config, "biomedclip_state_filename", None) is not None
    ):
        backbone_state_path = model_dir / config.biomedclip_state_filename
        if backbone_state_path.exists():
            state = torch.load(backbone_state_path, map_location=map_location)
            biomedclip_model.load_state_dict(state, strict=False)
            model.set_biomedclip_model(biomedclip_model)

    return model

In [21]:
export_dir = export_biomedclip_mlp_as_hf_pretrained(
    output_dir="/data/liangz2/openi/biomedclip_mimic_13label_cache/hf_pretrained_export",
    phase1_ckpt_path="/data/liangz2/openi/biomedclip_mimic_13label_cache/mlp_classifier_phase1_best.pt",
    phase2_ckpt_path="/data/liangz2/openi/biomedclip_mimic_13label_cache/mlp_classifier_phase2_best.pt",
    safe_serialization=False,   # set True if you want model.safetensors
)

✅ Hugging Face export completed: /data/liangz2/openi/biomedclip_mimic_13label_cache/hf_pretrained_export
   Files:
   - architecture.json
   - biomedclip_state.bin
   - config.json
   - label_names.json
   - processor_config.json
   - pytorch_model.bin


In [23]:
import torch
import torch.nn.functional as F
from PIL import Image
from typing import Dict, Any, Optional

import open_clip


def load_biomedclip_model_and_preprocess(
    model_name: str = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    """
    Load BiomedCLIP model + preprocess + tokenizer
    """
    model, _, preprocess = open_clip.create_model_and_transforms(model_name)
    tokenizer = open_clip.get_tokenizer(model_name)

    model = model.to(device)
    model.eval()

    return model, preprocess, tokenizer


def infer_cxr_biomedclip_mlp(
    model,  # your HF loaded model
    image_path: str,
    biomedclip_model,
    preprocess,
    tokenizer,
    text_prompt: Optional[str] = "chest x-ray report",
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> Dict[str, Any]:
    """
    Run inference on a single CXR image

    Returns:
        {
            "probabilities": {label: prob},
            "predictions": {label: 0/1},
            "top_labels": [(label, prob), ...]
        }
    """

    # ------------------------------------------------------------
    # 1. Load image
    # ------------------------------------------------------------
    image = Image.open(image_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    # ------------------------------------------------------------
    # 2. Prepare text (optional but recommended)
    # ------------------------------------------------------------
    if text_prompt is None:
        text_prompt = "chest x-ray"

    text_tokens = tokenizer([text_prompt]).to(device)

    # ------------------------------------------------------------
    # 3. Encode with BiomedCLIP
    # ------------------------------------------------------------
    with torch.no_grad():
        img_feat = biomedclip_model.encode_image(image_tensor).float()
        txt_feat = biomedclip_model.encode_text(text_tokens).float()

        img_feat = F.normalize(img_feat, dim=-1)
        txt_feat = F.normalize(txt_feat, dim=-1)

        fused = torch.cat([img_feat, txt_feat], dim=-1)

    # ------------------------------------------------------------
    # 4. Run classifier
    # ------------------------------------------------------------
    with torch.no_grad():
        outputs = model(fused_embeddings=fused)
        probs = outputs["probabilities"][0]  # shape [13]

    # ------------------------------------------------------------
    # 5. Convert to labels
    # ------------------------------------------------------------
    label_names = model.config.label_names
    threshold = model.config.threshold

    probs_list = probs.cpu().tolist()

    prob_dict = {
        label_names[i]: float(probs_list[i])
        for i in range(len(label_names))
    }

    pred_dict = {
        label_names[i]: int(probs_list[i] >= threshold)
        for i in range(len(label_names))
    }

    # sorted top labels
    top_labels = sorted(
        prob_dict.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return {
        "probabilities": prob_dict,
        "predictions": pred_dict,
        "top_labels": top_labels,
    }

In [25]:
# ------------------------------------------------------------
# Load HF model (already done)
# ------------------------------------------------------------
model = BiomedCLIPMLPForMultiLabelClassification.from_pretrained(
    "/data/liangz2/openi/biomedclip_mimic_13label_cache/hf_pretrained_export",
    local_files_only=True,
)
model.eval()
model.to("cuda")


# ------------------------------------------------------------
# Load BiomedCLIP
# ------------------------------------------------------------
biomedclip_model, preprocess, tokenizer = load_biomedclip_model_and_preprocess()

# ------------------------------------------------------------
# Run inference
# ------------------------------------------------------------

IMAGE_PATH = "/data/liangz2/openi/mimic_test/50022945.jpg"  # example image path

result = infer_cxr_biomedclip_mlp(
    model=model,
    image_path=IMAGE_PATH,
    biomedclip_model=biomedclip_model,
    preprocess=preprocess,
    tokenizer=tokenizer,
)

# Pretty print
import json
print(json.dumps(result, indent=2))

{
  "probabilities": {
    "atelectasis": 0.9179086089134216,
    "cardiomegaly": 0.7607906460762024,
    "consolidation": 0.15465761721134186,
    "edema": 0.18487995862960815,
    "enlarged cardiomediastinum": 0.8505334258079529,
    "fracture": 0.04000033810734749,
    "lung lesion": 0.01978853903710842,
    "lung opacity": 0.5231521129608154,
    "pleural effusion": 0.8950232267379761,
    "pleural other": 0.013289184309542179,
    "pneumonia": 0.23412393033504486,
    "pneumothorax": 0.7342323660850525,
    "support devices": 0.9228044152259827
  },
  "predictions": {
    "atelectasis": 1,
    "cardiomegaly": 1,
    "consolidation": 0,
    "edema": 0,
    "enlarged cardiomediastinum": 1,
    "fracture": 0,
    "lung lesion": 0,
    "lung opacity": 1,
    "pleural effusion": 1,
    "pleural other": 0,
    "pneumonia": 0,
    "pneumothorax": 1,
    "support devices": 1
  },
  "top_labels": [
    [
      "support devices",
      0.9228044152259827
    ],
    [
      "atelectasis",
  